In [4]:
"""
Fly a drone (quadrotor-shaped rigid body) from (0, 0, 1) to (3, 3, 3) in MuJoCo,
visualized with MuJoCo's built-in interactive 3D viewer.

Install requirements:
    pip install mujoco

Run:
    python mujoco_drone_flight.py

How it works
------------
- The drone is a single rigid body with a free joint (6 DOF: 3 translation + 3 rotation).
- Three "motor" actuators apply a world-frame force directly along the free joint's
  x/y/z translational DOFs (a simplified thrust model -- this skips individual rotor
  thrust mixing, which is unnecessary detail for a point-to-point flight demo).
- A minimum-jerk trajectory is generated from the start point to the goal point over
  a fixed duration, giving smooth position/velocity setpoints over time.
- A PD controller (with gravity feed-forward) tracks that trajectory by commanding
  the required world-frame force at every simulation step.
- After reaching the goal, the drone hovers in place.
"""

import time
import numpy as np
import mujoco
import mujoco.viewer

# ----------------------------------------------------------------------------
# 1. Build the drone model (MJCF)
# ----------------------------------------------------------------------------
# A central body + 4 arms + 4 rotor discs, purely for a quadrotor-like look.
# Physics is driven by 3 direct force actuators on the free joint (see below).

XML = """
<mujoco model="drone">
    <compiler angle="degree"/>
    <option timestep="0.002" gravity="0 0 -9.81"/>

    <visual>
        <headlight ambient="0.4 0.4 0.4" diffuse="0.6 0.6 0.6"/>
        <global offwidth="1280" offheight="720"/>
    </visual>

    <asset>
        <texture type="skybox" builtin="gradient" rgb1="0.65 0.8 0.95" rgb2="1 1 1" width="256" height="256"/>
        <texture name="grid" type="2d" builtin="checker" rgb1="0.75 0.8 0.75" rgb2="0.6 0.65 0.6" width="512" height="512"/>
        <material name="grid" texture="grid" texrepeat="8 8" reflectance="0.1"/>
    </asset>

    <worldbody>
        <light pos="0 0 6" dir="0 0 -1" diffuse="1 1 1" specular="0.2 0.2 0.2"/>
        <geom name="floor" type="plane" size="10 10 0.1" material="grid"/>

        <!-- markers for start and goal points -->
        <site name="start_marker" pos="0 0 1" size="0.06" rgba="0.1 0.8 0.1 0.6" type="sphere"/>
        <site name="goal_marker"  pos="3 3 3" size="0.06" rgba="0.9 0.1 0.1 0.6" type="sphere"/>

        <body name="drone" pos="0 0 1">
            <joint name="root" type="free"/>
            <inertial pos="0 0 0" mass="1.0" diaginertia="0.01 0.01 0.02"/>

            <!-- central hub -->
            <geom type="box" size="0.09 0.09 0.03" rgba="0.15 0.15 0.2 1" contype="0" conaffinity="0"/>

            <!-- 4 arms in an X configuration -->
            <geom type="capsule" fromto="0 0 0  0.22  0.22 0" size="0.012" rgba="0.2 0.2 0.25 1" contype="0" conaffinity="0"/>
            <geom type="capsule" fromto="0 0 0  0.22 -0.22 0" size="0.012" rgba="0.2 0.2 0.25 1" contype="0" conaffinity="0"/>
            <geom type="capsule" fromto="0 0 0 -0.22  0.22 0" size="0.012" rgba="0.2 0.2 0.25 1" contype="0" conaffinity="0"/>
            <geom type="capsule" fromto="0 0 0 -0.22 -0.22 0" size="0.012" rgba="0.2 0.2 0.25 1" contype="0" conaffinity="0"/>

            <!-- 4 rotor discs -->
            <geom type="cylinder" pos="0.22  0.22 0.01" size="0.09 0.005" rgba="0.1 0.5 0.9 0.5" contype="0" conaffinity="0"/>
            <geom type="cylinder" pos="0.22 -0.22 0.01" size="0.09 0.005" rgba="0.1 0.5 0.9 0.5" contype="0" conaffinity="0"/>
            <geom type="cylinder" pos="-0.22  0.22 0.01" size="0.09 0.005" rgba="0.9 0.3 0.1 0.5" contype="0" conaffinity="0"/>
            <geom type="cylinder" pos="-0.22 -0.22 0.01" size="0.09 0.005" rgba="0.9 0.3 0.1 0.5" contype="0" conaffinity="0"/>
        </body>
    </worldbody>

    <actuator>
        <!-- Direct world-frame force actuators along the free joint's translational DOFs.
             This is a simplified "ideal thrust vector" model -- fine for point-to-point
             flight demos where individual rotor mixing/attitude control isn't the point. -->
        <motor name="fx" joint="root" gear="1 0 0 0 0 0" ctrlrange="-30 30"/>
        <motor name="fy" joint="root" gear="0 1 0 0 0 0" ctrlrange="-30 30"/>
        <motor name="fz" joint="root" gear="0 0 1 0 0 0" ctrlrange="-30 30"/>
    </actuator>
</mujoco>
"""

# ----------------------------------------------------------------------------
# 2. Minimum-jerk trajectory generator
# ----------------------------------------------------------------------------
def min_jerk(p0, p1, t, T):
    """Return (position, velocity) along a minimum-jerk trajectory at time t in [0, T]."""
    t = np.clip(t / T, 0.0, 1.0)
    s = 10 * t**3 - 15 * t**4 + 6 * t**5          # position blend factor, 0 -> 1
    ds = (30 * t**2 - 60 * t**3 + 30 * t**4) / T   # d(blend)/dt
    pos = p0 + (p1 - p0) * s
    vel = (p1 - p0) * ds
    return pos, vel


# ----------------------------------------------------------------------------
# 3. Set up simulation
# ----------------------------------------------------------------------------
model = mujoco.MjModel.from_xml_string(XML)
data = mujoco.MjData(model)

start = np.array([0.0, 0.0, 1.0])
goal = np.array([3.0, 3.0, 3.0])
FLIGHT_TIME = 6.0   # seconds to travel from start to goal
HOVER_TIME = 3.0    # seconds to hover at the goal before the script exits

# Initial pose: start position, identity orientation, zero velocity
data.qpos[0:3] = start
data.qpos[3:7] = [1, 0, 0, 0]
data.qvel[:] = 0

drone_body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "drone")
mass = model.body_mass[drone_body_id]
g = -model.opt.gravity[2]

# PD gains (tuned for a light, damped drone)
Kp = np.array([6.0, 6.0, 10.0])
Kd = np.array([4.0, 4.0, 6.0])

# ----------------------------------------------------------------------------
# 4. Run with the interactive viewer
# ----------------------------------------------------------------------------
with mujoco.viewer.launch_passive(model, data) as viewer:
    # A nice initial camera angle looking at the flight path
    viewer.cam.lookat[:] = (start + goal) / 2
    viewer.cam.distance = 7
    viewer.cam.azimuth = 135
    viewer.cam.elevation = -20

    total_time = FLIGHT_TIME + HOVER_TIME
    print(f"Flying drone from {start} to {goal} over {FLIGHT_TIME:.1f}s, "
          f"then hovering for {HOVER_TIME:.1f}s...")

    last_print = -1.0
    while viewer.is_running() and data.time < total_time:
        step_start = time.time()

        # Desired position/velocity from the trajectory (clamped to goal after FLIGHT_TIME)
        pos_des, vel_des = min_jerk(start, goal, data.time, FLIGHT_TIME)

        pos = data.qpos[0:3]
        vel = data.qvel[0:3]

        pos_err = pos_des - pos
        vel_err = vel_des - vel

        # Desired acceleration (PD) + gravity feed-forward
        accel_des = Kp * pos_err + Kd * vel_err + np.array([0, 0, g])
        force = mass * accel_des

        data.ctrl[0:3] = force

        mujoco.mj_step(model, data)
        viewer.sync()

        if data.time - last_print > 0.5:
            dist = np.linalg.norm(goal - pos)
            print(f"t={data.time:5.1f}s  pos=({pos[0]:5.2f}, {pos[1]:5.2f}, {pos[2]:5.2f})  "
                  f"dist_to_goal={dist:5.2f}")
            last_print = data.time

        # keep roughly real-time
        elapsed = time.time() - step_start
        if elapsed < model.opt.timestep:
            time.sleep(model.opt.timestep - elapsed)

    print("Done. Final position:", data.qpos[0:3])


Flying drone from [0. 0. 1.] to [3. 3. 3.] over 6.0s, then hovering for 3.0s...
t=  0.0s  pos=( 0.00,  0.00,  1.00)  dist_to_goal= 4.69
t=  0.5s  pos=( 0.01,  0.01,  1.01)  dist_to_goal= 4.68
t=  1.0s  pos=( 0.07,  0.07,  1.05)  dist_to_goal= 4.58
t=  1.5s  pos=( 0.25,  0.25,  1.18)  dist_to_goal= 4.30
t=  2.0s  pos=( 0.56,  0.56,  1.39)  dist_to_goal= 3.81
t=  2.5s  pos=( 0.98,  0.98,  1.67)  dist_to_goal= 3.15
t=  3.0s  pos=( 1.47,  1.47,  1.99)  dist_to_goal= 2.39
t=  3.5s  pos=( 1.96,  1.96,  2.31)  dist_to_goal= 1.63
t=  4.0s  pos=( 2.40,  2.40,  2.60)  dist_to_goal= 0.94
t=  4.5s  pos=( 2.75,  2.75,  2.82)  dist_to_goal= 0.40
t=  5.0s  pos=( 2.97,  2.97,  2.96)  dist_to_goal= 0.06
t=  5.5s  pos=( 3.06,  3.06,  3.02)  dist_to_goal= 0.09
t=  6.0s  pos=( 3.06,  3.06,  3.02)  dist_to_goal= 0.08
t=  6.5s  pos=( 3.02,  3.02,  3.01)  dist_to_goal= 0.03
t=  7.0s  pos=( 3.01,  3.01,  3.00)  dist_to_goal= 0.01
t=  7.5s  pos=( 3.00,  3.00,  3.00)  dist_to_goal= 0.00
t=  8.0s  pos=( 3.00,  3